In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q chromadb sentence-transformers


Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 79.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 107.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 84.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are install

In [ ]:
import chromadb
import sentence_transformers

print("chromadb", chromadb.__version__)
print("sentence-transformers OK")


chromadb 1.5.9
sentence-transformers OK


In [ ]:
from pathlib import Path

DRIVE_ROOT = Path("/content/drive/MyDrive")

CHUNK_ROOT = DRIVE_ROOT / "chunk"
OUTPUT_DB_DIR = DRIVE_ROOT / "doctrine_chroma_db"
OUTPUT_ZIP = DRIVE_ROOT / "doctrine_chroma_db.zip"

MODEL_NAME = "BAAI/bge-m3"
BATCH_SIZE = 64

BRANCH_CONFIG = {
    "army": {
        "label": "육군",
        "chunk_dir": CHUNK_ROOT / "army_chunk",
        "collection": "army_doctrine",
    },
    "navy": {
        "label": "해군",
        "chunk_dir": CHUNK_ROOT / "navy_chunk",
        "collection": "navy_doctrine",
    },
    "airforce": {
        "label": "공군",
        "chunk_dir": CHUNK_ROOT / "air_force_chunk",
        "collection": "airforce_doctrine",
    },
}
print("완료")

완료


In [ ]:
import csv
from collections import defaultdict

def choose_core_csv(csv_files):
    corrected = [
        path for path in csv_files
        if "all_rag_chunks" in path.name.lower() and "corrected" in path.name.lower()
    ]
    if corrected:
        return max(corrected, key=lambda path: path.stat().st_size)

    all_chunks = [path for path in csv_files if "all_rag_chunks" in path.name.lower()]
    if all_chunks:
        return max(all_chunks, key=lambda path: path.stat().st_size)

    metadata = [path for path in csv_files if "chunk_metadata" in path.name.lower()]
    if metadata:
        return max(metadata, key=lambda path: path.stat().st_size)

    return max(csv_files, key=lambda path: path.stat().st_size) if csv_files else None


def read_header_and_count(path):
    with path.open("r", encoding="utf-8-sig", newline="") as f:
        reader = csv.reader(f)
        rows = [row for row in reader if row and any(cell.strip() for cell in row)]
    if not rows:
        return [], 0
    return [cell.strip() for cell in rows[0]], max(0, len(rows) - 1)


def inspect_branch(branch, config):
    root = config["chunk_dir"]
    reports = []

    if not root.exists():
        raise FileNotFoundError(f"{branch} chunk folder not found: {root}")

    folders = sorted([path for path in root.iterdir() if path.is_dir()])

    for folder in folders:
        csv_files = sorted(folder.rglob("*.csv"))
        core_csv = choose_core_csv(csv_files)
        header, row_count = read_header_and_count(core_csv) if core_csv else ([], 0)
        header_set = set(header)

        required_groups = {
            "chunk_id": {"chunk_id", "rag_chunk_id"},
            "document_id": {"document_id"},
            "document_title": {"document_title", "document_short_title"},
            "text": {"embedding_text", "chunk_text", "text"},
            "citation": {"citation_text"},
            "page": {"pdf_page_start"},
        }

        missing = [
            name for name, candidates in required_groups.items()
            if not header_set.intersection(candidates)
        ]

        status = "ok"
        if not core_csv:
            status = "no_csv"
        elif row_count == 0:
            status = "empty_core_csv"
        elif missing:
            status = "missing_required_columns"

        reports.append({
            "branch": branch,
            "folder": folder.name,
            "status": status,
            "rows": row_count,
            "core_csv": core_csv,
            "core_csv_name": core_csv.name if core_csv else "",
            "missing": missing,
        })

    return reports


all_reports = []

for branch, config in BRANCH_CONFIG.items():
    reports = inspect_branch(branch, config)
    all_reports.extend(reports)

    print(f"\n[{branch}] folders={len(reports)}")
    for item in reports:
        print(
            f"- {item['folder']}: "
            f"{item['status']}, rows={item['rows']}, core={item['core_csv_name']}"
        )

bad = [item for item in all_reports if item["status"] != "ok"]

if bad:
    raise RuntimeError(f"Precheck failed: {bad}")

print("\nPrecheck OK")
print("total folders:", len(all_reports))
print("total rows:", sum(item["rows"] for item in all_reports))



[army] folders=10
- FM2_0_2023_RAG_Chunk_Output: ok, rows=1685, core=FM2_0_2023_All_RAG_Chunks_corrected.csv
- FM3_09_2024_RAG_Chunk_Output: ok, rows=1542, core=FM3_09_2024_All_RAG_Chunks_corrected.csv
- FM3_0_2025_RAG_Chunk_Output: ok, rows=1512, core=FM3_0_2025_All_RAG_Chunks_corrected.csv
- FM3_60_2023_RAG_Chunk_Output: ok, rows=918, core=FM3_60_2023_All_RAG_Chunks_corrected.csv
- FM3_90_2023_RAG_Chunk_Output: ok, rows=2696, core=FM3_90_2023_All_RAG_Chunks_corrected.csv
- FM3_94_2021_RAG_Chunk_Output: ok, rows=931, core=FM3_94_2021_All_RAG_Chunks_corrected.csv
- FM3_96_2021_RAG_Chunk_Output: ok, rows=1488, core=FM3_96_2021_All_RAG_Chunks_corrected.csv
- FM4_0_2026_RAG_Chunk_Output: ok, rows=1171, core=FM4_0_2026_All_RAG_Chunks.csv
- FM5_0_2024_C1_RAG_Chunk_Output: ok, rows=1879, core=FM5_0_2024_C1_All_RAG_Chunks_corrected.csv
- FM6_0_2022_RAG_Chunk_Output: ok, rows=1452, core=FM6_0_2022_All_RAG_Chunks_corrected.csv

[navy] folders=10
- JP3_02_2009_RAG_Chunk_Output: ok, rows=1731, c

In [ ]:
import json

TEXT_FIELDS = ["embedding_text", "chunk_text", "text", "summary_ko"]
CHUNK_ID_FIELDS = ["chunk_id", "rag_chunk_id"]


def read_csv_dicts(path):
    with path.open("r", encoding="utf-8-sig", newline="") as f:
        reader = csv.DictReader(f)
        return [{str(k).strip(): (v or "").strip() for k, v in row.items()} for row in reader]


def first_present(row, fields):
    for field in fields:
        value = row.get(field, "").strip()
        if value:
            return value
    return ""


def clean_metadata_value(value):
    value = "" if value is None else str(value).strip()
    return value


def build_record(row, branch, branch_label, folder_name, core_csv_name, row_index):
    document_id = row.get("document_id", "").strip() or folder_name.replace("_RAG_Chunk_Output", "")
    raw_chunk_id = first_present(row, CHUNK_ID_FIELDS) or f"row_{row_index:06d}"

    record_id = f"{branch}::{document_id}::{raw_chunk_id}"

    document = first_present(row, TEXT_FIELDS)
    if not document:
        return None

    chunk_text = row.get("chunk_text", "").strip() or row.get("text", "").strip() or document
    citation_text = row.get("citation_text", "").strip()

    if not citation_text:
        page = row.get("pdf_page_start", "").strip()
        citation_text = f"{row.get('document_short_title') or document_id}, PDF p.{page}".strip()

    metadata = {
        "branch": branch,
        "branch_label": branch_label,
        "chunk_id": record_id,
        "raw_chunk_id": raw_chunk_id,
        "document_id": document_id,
        "document_title": row.get("document_title", "").strip(),
        "document_short_title": row.get("document_short_title", "").strip() or document_id,
        "source_file": row.get("source_file", "").strip(),
        "source_folder": folder_name,
        "source_csv": core_csv_name,
        "pdf_page_start": row.get("pdf_page_start", "").strip(),
        "pdf_page_end": row.get("pdf_page_end", "").strip(),
        "printed_page_start": row.get("printed_page_start", "").strip(),
        "printed_page_end": row.get("printed_page_end", "").strip(),
        "chapter": row.get("chapter", "").strip(),
        "section": row.get("section", "").strip(),
        "subsection": row.get("subsection", "").strip(),
        "chunk_type": row.get("chunk_type", "").strip(),
        "retrieval_priority": row.get("retrieval_priority", "").strip(),
        "content_function": row.get("content_function", "").strip(),
        "mode_relevance": row.get("mode_relevance", "").strip(),
        "keywords_en": row.get("keywords_en", "").strip(),
        "keywords_ko": row.get("keywords_ko", "").strip(),
        "summary_ko": row.get("summary_ko", "").strip(),
        "citation_text": citation_text,
        "security_class": row.get("security_class", "").strip(),
        "processing_status": row.get("processing_status", "").strip(),
        "review_required": row.get("review_required", "").strip(),
        "version": row.get("version", "").strip(),
    }

    metadata = {key: clean_metadata_value(value) for key, value in metadata.items()}

    return {
        "id": record_id,
        "document": document,
        "chunk_text": chunk_text,
        "metadata": metadata,
    }


def load_branch_records(branch, config, reports):
    records = []
    seen_ids = set()
    duplicate_count = 0
    skipped_empty = 0

    for report in reports:
        rows = read_csv_dicts(report["core_csv"])

        for row_index, row in enumerate(rows, 1):
            record = build_record(
                row=row,
                branch=branch,
                branch_label=config["label"],
                folder_name=report["folder"],
                core_csv_name=report["core_csv_name"],
                row_index=row_index,
            )

            if record is None:
                skipped_empty += 1
                continue

            record_id = record["id"]

            if record_id in seen_ids:
                duplicate_count += 1
                record["id"] = f"{record_id}::dup_{duplicate_count:05d}"
                record["metadata"]["chunk_id"] = record["id"]

            seen_ids.add(record["id"])
            records.append(record)

    return records, duplicate_count, skipped_empty


records_by_branch = {}

for branch, config in BRANCH_CONFIG.items():
    reports = [item for item in all_reports if item["branch"] == branch]
    records, duplicate_count, skipped_empty = load_branch_records(branch, config, reports)

    records_by_branch[branch] = records

    print(
        f"{branch}: records={len(records)}, "
        f"duplicate_fixed={duplicate_count}, skipped_empty={skipped_empty}"
    )

print("all records:", sum(len(items) for items in records_by_branch.values()))


army: records=15274, duplicate_fixed=0, skipped_empty=0
navy: records=8483, duplicate_fixed=838, skipped_empty=0
airforce: records=2519, duplicate_fixed=0, skipped_empty=0
all records: 26276


In [ ]:
import time
import chromadb
from sentence_transformers import SentenceTransformer


def iter_batches(items, size):
    for start in range(0, len(items), size):
        yield start, items[start:start + size]


def get_existing_ids(collection):
    count = collection.count()
    if count == 0:
        return set()

    existing = set()
    page_size = 5000

    for offset in range(0, count, page_size):
        batch = collection.get(limit=page_size, offset=offset, include=[])
        existing.update(str(item_id) for item_id in batch.get("ids", []))

    return existing


print("Loading embedding model:", MODEL_NAME)
model = SentenceTransformer(MODEL_NAME)

client = chromadb.PersistentClient(path=str(OUTPUT_DB_DIR))

for branch, config in BRANCH_CONFIG.items():
    collection_name = config["collection"]

    print(f"\n=== Building collection: {collection_name} ===")

    existing_names = {collection.name for collection in client.list_collections()}

    if collection_name in existing_names:
        collection = client.get_collection(collection_name)
        print("existing_count:", collection.count())
    else:
        collection = client.create_collection(
            name=collection_name,
            metadata={
                "branch": branch,
                "branch_label": config["label"],
                "embedding_model": MODEL_NAME,
                "schema": "joint_doctrine_branch_collection_v1",
            },
        )

    existing_ids = get_existing_ids(collection)
    records = records_by_branch[branch]
    pending = [record for record in records if record["id"] not in existing_ids]

    print(
        f"records={len(records)}, "
        f"existing={len(existing_ids)}, pending={len(pending)}"
    )

    start_time = time.perf_counter()

    for start, batch in iter_batches(pending, BATCH_SIZE):
        ids = [item["id"] for item in batch]
        documents = [item["document"] for item in batch]
        metadatas = [item["metadata"] for item in batch]

        embeddings = model.encode(
            documents,
            normalize_embeddings=True,
            show_progress_bar=False,
            batch_size=min(BATCH_SIZE, 32),
        ).tolist()

        collection.add(
            ids=ids,
            documents=documents,
            metadatas=metadatas,
            embeddings=embeddings,
        )

        done = len(existing_ids) + start + len(batch)
        elapsed = time.perf_counter() - start_time

        print(
            f"{collection_name}: added {done}/{len(records)} "
            f"elapsed_sec={elapsed:.1f}",
            flush=True,
        )

    final_count = collection.count()

    print(f"{collection_name}: final_count={final_count}")

    if final_count != len(records):
        raise RuntimeError(
            f"Count mismatch in {collection_name}: "
            f"{final_count} != {len(records)}"
        )

print("\nAll collections built successfully.")


Loading embedding model: BAAI/bge-m3


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]


=== Building collection: army_doctrine ===
records=15274, existing=0, pending=15274
army_doctrine: added 64/15274 elapsed_sec=6.1
army_doctrine: added 128/15274 elapsed_sec=10.0
army_doctrine: added 192/15274 elapsed_sec=14.0
army_doctrine: added 256/15274 elapsed_sec=18.3
army_doctrine: added 320/15274 elapsed_sec=22.4
army_doctrine: added 384/15274 elapsed_sec=26.8
army_doctrine: added 448/15274 elapsed_sec=31.1
army_doctrine: added 512/15274 elapsed_sec=35.7
army_doctrine: added 576/15274 elapsed_sec=39.9
army_doctrine: added 640/15274 elapsed_sec=44.4
army_doctrine: added 704/15274 elapsed_sec=49.0
army_doctrine: added 768/15274 elapsed_sec=53.9
army_doctrine: added 832/15274 elapsed_sec=58.4
army_doctrine: added 896/15274 elapsed_sec=63.7
army_doctrine: added 960/15274 elapsed_sec=68.3
army_doctrine: added 1024/15274 elapsed_sec=74.1
army_doctrine: added 1088/15274 elapsed_sec=79.4
army_doctrine: added 1152/15274 elapsed_sec=84.3
army_doctrine: added 1216/15274 elapsed_sec=88.9
a

OutOfMemoryError: CUDA out of memory. Tried to allocate 5.35 GiB. GPU 0 has a total capacity of 14.56 GiB of which 2.27 GiB is free. Including non-PyTorch memory, this process has 12.29 GiB memory in use. Of the allocated memory 12.09 GiB is allocated by PyTorch, and 78.79 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
import gc
import torch

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("memory cleared")


memory cleared


In [ ]:
import gc
import time
import chromadb
import torch
from sentence_transformers import SentenceTransformer

# 메모리 정리
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("memory cleared")

# 배치 크기 조정
BATCH_SIZE = 16
ENCODE_BATCH_SIZE = 8

print("BATCH_SIZE:", BATCH_SIZE)
print("ENCODE_BATCH_SIZE:", ENCODE_BATCH_SIZE)


def iter_batches(items, size):
    for start in range(0, len(items), size):
        yield start, items[start:start + size]


def get_existing_ids(collection):
    count = collection.count()

    if count == 0:
        return set()

    existing = set()
    page_size = 5000

    for offset in range(0, count, page_size):
        batch = collection.get(
            limit=page_size,
            offset=offset,
            include=[]
        )
        existing.update(str(item_id) for item_id in batch.get("ids", []))

    return existing


print("Loading embedding model:", MODEL_NAME)

# 이미 model 변수가 있으면 재사용하고, 없으면 새로 로드
try:
    model
    print("Using existing loaded model")
except NameError:
    model = SentenceTransformer(MODEL_NAME)
    print("Model loaded")

client = chromadb.PersistentClient(path=str(OUTPUT_DB_DIR))

for branch, config in BRANCH_CONFIG.items():
    collection_name = config["collection"]

    print(f"\n=== Building collection: {collection_name} ===")

    existing_names = {collection.name for collection in client.list_collections()}

    if collection_name in existing_names:
        collection = client.get_collection(collection_name)
        print("existing_count:", collection.count())
    else:
        collection = client.create_collection(
            name=collection_name,
            metadata={
                "branch": branch,
                "branch_label": config["label"],
                "embedding_model": MODEL_NAME,
                "schema": "joint_doctrine_branch_collection_v1",
            },
        )
        print("collection created:", collection_name)

    existing_ids = get_existing_ids(collection)

    records = records_by_branch[branch]
    pending = [
        record for record in records
        if record["id"] not in existing_ids
    ]

    print(
        f"records={len(records)}, "
        f"existing={len(existing_ids)}, "
        f"pending={len(pending)}"
    )

    if not pending:
        final_count = collection.count()
        print(f"{collection_name}: already complete, final_count={final_count}")
        continue

    start_time = time.perf_counter()

    for start, batch in iter_batches(pending, BATCH_SIZE):
        ids = [item["id"] for item in batch]
        documents = [item["document"] for item in batch]
        metadatas = [item["metadata"] for item in batch]

        embeddings = model.encode(
            documents,
            normalize_embeddings=True,
            show_progress_bar=False,
            batch_size=ENCODE_BATCH_SIZE,
        ).tolist()

        collection.add(
            ids=ids,
            documents=documents,
            metadatas=metadatas,
            embeddings=embeddings,
        )

        done = len(existing_ids) + start + len(batch)
        elapsed = time.perf_counter() - start_time

        print(
            f"{collection_name}: added {done}/{len(records)} "
            f"elapsed_sec={elapsed:.1f}",
            flush=True,
        )

        # 매 배치마다 메모리 정리
        del embeddings
        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    final_count = collection.count()

    print(f"{collection_name}: final_count={final_count}")

    if final_count != len(records):
        raise RuntimeError(
            f"Count mismatch in {collection_name}: "
            f"{final_count} != {len(records)}"
        )

print("\nAll collections built successfully.")


memory cleared
BATCH_SIZE: 16
ENCODE_BATCH_SIZE: 8
Loading embedding model: BAAI/bge-m3
Using existing loaded model

=== Building collection: army_doctrine ===
existing_count: 4544
records=15274, existing=4544, pending=10730
army_doctrine: added 4560/15274 elapsed_sec=25.3
army_doctrine: added 4576/15274 elapsed_sec=26.2
army_doctrine: added 4592/15274 elapsed_sec=27.1
army_doctrine: added 4608/15274 elapsed_sec=27.9
army_doctrine: added 4624/15274 elapsed_sec=28.7
army_doctrine: added 4640/15274 elapsed_sec=29.6
army_doctrine: added 4656/15274 elapsed_sec=30.5
army_doctrine: added 4672/15274 elapsed_sec=31.3
army_doctrine: added 4688/15274 elapsed_sec=32.2
army_doctrine: added 4704/15274 elapsed_sec=33.1
army_doctrine: added 4720/15274 elapsed_sec=33.9
army_doctrine: added 4736/15274 elapsed_sec=35.7
army_doctrine: added 4752/15274 elapsed_sec=36.8
army_doctrine: added 4768/15274 elapsed_sec=37.8
army_doctrine: added 4784/15274 elapsed_sec=39.2
army_doctrine: added 4800/15274 elapsed_

In [ ]:
def verify_collection(collection_name, queries):
    collection = client.get_collection(collection_name)

    embeddings = model.encode(
        queries,
        normalize_embeddings=True,
        show_progress_bar=False,
        batch_size=4,
    ).tolist()

    result = collection.query(
        query_embeddings=embeddings,
        n_results=5,
        include=["documents", "metadatas", "distances"],
    )

    print(f"\n### {collection_name}")
    print("count:", collection.count())

    for q_index, query in enumerate(queries):
        print("\nquery:", query)

        for rank, doc_id in enumerate(result["ids"][q_index], 1):
            metadata = result["metadatas"][q_index][rank - 1]
            distance = result["distances"][q_index][rank - 1]

            print(
                f"{rank}. distance={distance:.4f} "
                f"branch={metadata.get('branch')} "
                f"doc={metadata.get('document_short_title')} "
                f"page={metadata.get('pdf_page_start')} "
                f"source={metadata.get('source_folder')}"
            )


VERIFY_QUERIES = [
    "작전 계획 수립 시 지휘관과 참모가 검토해야 할 핵심 요소는 무엇인가?",
    "화력지원 요청 시 검토해야 할 지휘통제 및 조정 사항은 무엇인가?",
]

verify_collection("army_doctrine", VERIFY_QUERIES)
verify_collection("navy_doctrine", VERIFY_QUERIES)
verify_collection("airforce_doctrine", VERIFY_QUERIES)



### army_doctrine
count: 15274

query: 작전 계획 수립 시 지휘관과 참모가 검토해야 할 핵심 요소는 무엇인가?
1. distance=0.6601 branch=army doc=FM 5-0 page=114 source=FM5_0_2024_C1_RAG_Chunk_Output
2. distance=0.6845 branch=army doc=FM 5-0 page=114 source=FM5_0_2024_C1_RAG_Chunk_Output
3. distance=0.7078 branch=army doc=FM 5-0 page=32 source=FM5_0_2024_C1_RAG_Chunk_Output
4. distance=0.7156 branch=army doc=FM 5-0 page=20 source=FM5_0_2024_C1_RAG_Chunk_Output
5. distance=0.7170 branch=army doc=FM 5-0 page=105 source=FM5_0_2024_C1_RAG_Chunk_Output

query: 화력지원 요청 시 검토해야 할 지휘통제 및 조정 사항은 무엇인가?
1. distance=0.7092 branch=army doc=FM 3-09 page=71 source=FM3_09_2024_RAG_Chunk_Output
2. distance=0.7376 branch=army doc=FM 4-0 page=159 source=FM4_0_2026_RAG_Chunk_Output
3. distance=0.7398 branch=army doc=FM 3-09 page=244 source=FM3_09_2024_RAG_Chunk_Output
4. distance=0.7554 branch=army doc=FM 4-0 page=120 source=FM4_0_2026_RAG_Chunk_Output
5. distance=0.7629 branch=army doc=FM 4-0 page=142 source=FM4_0_2026_RAG_Chunk_Output

In [ ]:
import shutil
from google.colab import files

# 기존 zip이 있으면 삭제
if OUTPUT_ZIP.exists():
    OUTPUT_ZIP.unlink()

# doctrine_chroma_db 폴더를 zip으로 압축
archive_base = str(OUTPUT_ZIP).replace(".zip", "")
shutil.make_archive(
    archive_base,
    "zip",
    root_dir=str(OUTPUT_DB_DIR)
)

print("zip created:", OUTPUT_ZIP)
print("size MB:", OUTPUT_ZIP.stat().st_size / 1024 / 1024)

# 브라우저로 직접 다운로드
files.download(str(OUTPUT_ZIP))


zip created: /content/drive/MyDrive/doctrine_chroma_db.zip
size MB: 203.0488576889038


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>